# 04 — Video Model (CNN + LSTM)
Extracts 16 frames per video from the UCF-10 dataset, trains a ResNet50 + LSTM
model for action recognition.

**Dataset:** UCF-10 subset (10 action classes from UCF-101)

**Pipeline:**
1. Extract 16 evenly-spaced frames per video → save as JPEGs
2. Feed frame sequence through ResNet50 (CNN) → get 2048-d features per frame
3. Feed the 16-frame feature sequence through LSTM → get temporal representation
4. Classify with MLP head

## 1. Mount Drive and copy dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Copy video dataset from Drive to fast local SSD
if not os.path.exists('/content/video_dataset'):
    print('Copying video dataset from Drive...')
    os.system('cp -r /content/drive/MyDrive/ContentRecognition/video_dataset /content/')
    print('Done!')
else:
    print('Video dataset already on local SSD.')

## 2. Imports

In [ ]:
!pip install opencv-python-headless -q

import os, time, random
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 3. Config

In [ ]:
CLASS_NAMES = [
    'Basketball', 'Biking', 'Bowling', 'CliffDiving',
    'GolfSwing', 'HorseRiding', 'Skiing', 'Surfing',
    'TennisSwing', 'SkateBoarding'
]
NUM_CLASSES      = len(CLASS_NAMES)
FRAMES_PER_VIDEO = 16
BATCH_SIZE       = 8
EPOCHS           = 10

VIDEO_DIR  = '/content/video_dataset'
FRAMES_DIR = '/content/ucf_frames'
BASE_DIR   = '/content/drive/MyDrive/ContentRecognition'
CKPT_DIR   = f'{BASE_DIR}/checkpoints/video'
RESULTS_DIR= f'{BASE_DIR}/results/video'

os.makedirs(FRAMES_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Config ready.')

## 4. Verify dataset classes
Confirms the dataset folder has all 10 expected classes.

In [ ]:
found = sorted(os.listdir(VIDEO_DIR))
print(f'Found {len(found)} classes:')
for i, c in enumerate(found):
    marker = '✅' if c in CLASS_NAMES else '⚠️  (unexpected)'
    print(f'  {i+1:2}. {c}  {marker}')

## 5. Frame extraction
For each video: pick 16 evenly-spaced frame indices → resize to 224×224 → save as JPEGs.
Each video gets its own subfolder inside `ucf_frames/<class>/<video_name>/`.

**Skip this cell if frames are already extracted.**

In [ ]:
def extract_frames(video_path, output_folder, num_frames=FRAMES_PER_VIDEO):
    """Extract `num_frames` evenly-spaced frames from a video file."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0:
        cap.release()
        return False

    indices = set(np.linspace(0, total - 1, num_frames, dtype=int))
    saved, frame_idx = 0, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in indices:
            frame = cv2.resize(frame, (224, 224))
            cv2.imwrite(os.path.join(output_folder, f'frame_{saved:03d}.jpg'), frame)
            saved += 1
        frame_idx += 1

    cap.release()
    return saved == num_frames


total_success, total_videos = 0, 0
for cls in CLASS_NAMES:
    cls_video_dir  = os.path.join(VIDEO_DIR, cls)
    cls_frames_dir = os.path.join(FRAMES_DIR, cls)
    os.makedirs(cls_frames_dir, exist_ok=True)

    videos = [v for v in os.listdir(cls_video_dir)
              if v.endswith(('.avi', '.mp4', '.mov'))]
    print(f'\n{cls} — {len(videos)} videos')

    success = 0
    for video in tqdm(videos):
        video_path  = os.path.join(cls_video_dir, video)
        video_name  = os.path.splitext(video)[0]
        output_dir  = os.path.join(cls_frames_dir, video_name)
        os.makedirs(output_dir, exist_ok=True)

        # Skip if already extracted
        if len(os.listdir(output_dir)) == FRAMES_PER_VIDEO:
            success += 1
            continue

        if extract_frames(video_path, output_dir):
            success += 1

    print(f'  Extracted: {success}/{len(videos)}')
    total_success += success
    total_videos  += len(videos)

print(f'\nTotal: {total_success}/{total_videos} videos successfully extracted.')

## 6. Build dataset samples list

In [ ]:
class_to_idx = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

all_samples = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(FRAMES_DIR, cls)
    if not os.path.exists(cls_path):
        print(f'WARNING: missing class folder: {cls}')
        continue
    for video_name in os.listdir(cls_path):
        video_path = os.path.join(cls_path, video_name)
        # Only include videos with exactly FRAMES_PER_VIDEO frames
        if os.path.isdir(video_path) and len(os.listdir(video_path)) == FRAMES_PER_VIDEO:
            all_samples.append((video_path, class_to_idx[cls]))

random.seed(42)          # Fixed seed → reproducible split
random.shuffle(all_samples)
split          = int(0.8 * len(all_samples))
train_samples  = all_samples[:split]
val_samples    = all_samples[split:]

print(f'Total valid videos : {len(all_samples)}')
print(f'Train              : {len(train_samples)}')
print(f'Val                : {len(val_samples)}')

## 7. Dataset and DataLoader

In [ ]:
class VideoFrameDataset(Dataset):
    """
    Loads 16 saved JPEG frames for each video.
    Returns (video_tensor, label) where video_tensor shape = (16, 3, 224, 224).
    """
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frame_files = sorted(os.listdir(video_path))   # sorted → consistent order

        frames = []
        for fname in frame_files:
            img = Image.open(os.path.join(video_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)

        # Stack: list of (3,224,224) → (16,3,224,224)
        return torch.stack(frames), label


# Transforms — no augmentation for validation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.3,
        hue=0.1
    ),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = VideoFrameDataset(train_samples, transform=train_transform)
val_dataset   = VideoFrameDataset(val_samples,   transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

## 8. CNN + LSTM model
- **CNN (ResNet50):** extracts spatial features from each frame → (2048,)
- **LSTM:** models temporal dependencies across the 16-frame sequence
- Only `layer4` of ResNet is unfrozen — fine-tunes the highest-level visual features
  while keeping the rest of the backbone stable (faster training, less overfitting).

In [ ]:
class CNN_LSTM(nn.Module):
    """
    Per-frame CNN (ResNet50, layer4 unfrozen) + LSTM temporal model.
    Input  : (batch, 16, 3, 224, 224)
    Output : (batch, num_classes)
    """
    def __init__(self, num_classes=10, hidden_size=512, num_layers=2):
        super().__init__()

        # Load pretrained ResNet50
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # Freeze all layers first
        for p in resnet.parameters():
            p.requires_grad = False

        # Unfreeze only layer4 (highest-level spatial features)
        for name, p in resnet.named_parameters():
            if 'layer4' in name:
                p.requires_grad = True

        # Remove the FC classification head → output is (B*T, 2048, 1, 1)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        # LSTM: processes 16-frame feature sequence
        self.lstm = nn.LSTM(
            input_size  = 2048,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = 0.3
        )

        # Classifier: takes last LSTM hidden state
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.6),      # increased from 0.4
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),      # extra dropout layer
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (batch, frames, C, H, W)
        B, T, C, H, W = x.shape

        # Merge batch and time dims → process all frames through CNN at once
        x = x.view(B * T, C, H, W)                  # (B*T, C, H, W)
        cnn_feat = self.cnn(x)                       # (B*T, 2048, 1, 1)
        cnn_feat = cnn_feat.view(B, T, -1)           # (B, T, 2048)

        # LSTM over time
        lstm_out, _ = self.lstm(cnn_feat)            # (B, T, hidden_size)
        last        = lstm_out[:, -1, :]             # (B, hidden_size) — last frame

        return self.classifier(last)                 # (B, num_classes)


model = CNN_LSTM(num_classes=NUM_CLASSES).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CNN+LSTM ready!  Trainable params: {trainable:,}')

## 9. Train

In [ ]:
criterion   = nn.CrossEntropyLoss()
optimizer   = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
)
# Reduce LR by 10x every 5 epochs
scheduler   = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

history     = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
ckpt_path   = f'{CKPT_DIR}/cnn_lstm_best.pth'

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 42)
    t0 = time.time()

    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        loader = train_loader if phase == 'train' else val_loader

        running_loss, running_correct = 0.0, 0

        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs  = model(inputs)
                loss     = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss    += loss.item() * inputs.size(0)
            running_correct += torch.sum(preds == labels)

        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc  = running_correct.double() / len(loader.dataset)

        history[f'{phase}_loss'].append(epoch_loss)
        history[f'{phase}_acc'].append(epoch_acc.item())
        print(f'  {phase.upper():5} → Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_val_acc:
            best_val_acc = epoch_acc
            torch.save(model.state_dict(), ckpt_path)
            print(f'  ✅ Best saved! Val Acc: {best_val_acc:.4f}')

    print(f'  ⏱  {time.time()-t0:.1f}s')

print(f'\nTraining complete! Best Val Acc: {best_val_acc:.4f}')

## 10. Evaluate and plot

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        outputs  = model(inputs.to(device))
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

axes[0].plot(history['train_acc'],  label='Train', marker='o', color='steelblue')
axes[0].plot(history['val_acc'],    label='Val',   marker='o', color='darkorange')
axes[0].set_title('CNN+LSTM Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train', marker='o', color='steelblue')
axes[1].plot(history['val_loss'],   label='Val',   marker='o', color='darkorange')
axes[1].set_title('CNN+LSTM Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[2])
axes[2].set_title('Confusion Matrix', fontweight='bold')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
plt.setp(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cnn_lstm_results.png', dpi=150)
plt.show()
print('Results saved!')